# Common Libraries

In [1]:
import os, sys, shutil, pickle
import numpy as np
import matplotlib.pylab as plt

import opensim
if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco

# Custom Libraries

In [2]:
myoconverter_path = "/home/seojin/biomechanics_tools/myoconverter"
sys.path.append(myoconverter_path)
from myoconverter.optimization.utils.UtilsMujoco import getMuscleForceLengthCurvesSim, getCoordinateRange_mjc
from myoconverter.optimization.model_states.OsimMuscleStates import OsimMuscleStates
from myoconverter.optimization.utils.UtilsForceOpt import getMuscleForceDiff, fmOptPSO_cust
from myoconverter.optimization.utils.UtilsOpensim import getJointsControlledByMuscle, getCouplingJoints, \
    getMuscleProperties, getJointRanges_array, getMuscleLengthList, getMuscleForceMaps


# Params

In [3]:
speedy = True
is_update = False

geometry_dir_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_emg/Myosuite/scaled_model/Geometry"
cv2_model_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_emg/Myosuite/scaled_model/scaled_model_cvt2.xml"
osim_model_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_emg/Myosuite/scaled_model/OpenSim/scaled_model.osim"

output_dir_path = "/mnt/ext1/seojin/temp/myoconverter"
save_model_path = os.path.join(output_dir_path, "cvt3.xml")

muscle_name = "TMIN"

In [4]:
muscle_path = os.path.join(output_dir_path, f"{muscle_name}.pkl")

# Load model

In [5]:
# Mujoco
mjc_model = mujoco.MjModel.from_xml_path(cv2_model_path)

# OpenSim
osim_model = opensim.Model(osim_model_path)
osim_muscles = osim_model.getMuscles()
osim_muscle = osim_muscles.get(muscle_name)

[info] Loaded model setup_scale from file /mnt/sdb2/DeepProprioception/Projects/DP01_emg/Myosuite/scaled_model/OpenSim/scaled_model.osim


# Constants

In [6]:
# Joint
ang_ranges_mjc, free_jnt_id_mjc = getCoordinateRange_mjc(mjc_model)

# Muscle
muscle_names = [muscle.getName() for muscle in osim_model.getMuscles()]
muscle_i = muscle_names.index(muscle_name)

# Copy geometry

In [7]:
os.system(f"cp -r {geometry_dir_path} {output_dir_path}")

0

# OpenSim

## OpenSim Wraping coords

Extract the wrapping coordinates of the given muscle list. In addition, the coupling coordinates when calculating moment arms were extracted.

In [8]:
wrapping_coordinate = {}

# get wrapping coordiantes
joints = getJointsControlledByMuscle(osim_model, osim_muscle)
jointCP = getCouplingJoints(osim_model, osim_muscle, joints)

# remove the duplicated coupling joints combinations
wrap_list = [list(i) for i in {*[tuple(sorted(i)) for i in jointCP]}]

"""
remove the sub coupling joints and only save the largest one.
Also make sure they always have the same order in every calculations.
A dirty fix right now, must have better ways to do this....
"""
wrapping_coordinate = {}
i_wrap = 0
while i_wrap < len(wrap_list):
    j_wrap = 0
    while j_wrap < len(wrap_list):
        if i_wrap == j_wrap:
            j_wrap = j_wrap + 1
        else:
            if all(elem in wrap_list[j_wrap] for elem in wrap_list[i_wrap]):
                del wrap_list[i_wrap]
                break
            else:
                j_wrap = j_wrap + 1
    else:
        i_wrap = i_wrap + 1
        
wrapping_coordinate[muscle_name] = wrap_list

In [9]:
# get a list of muscle lengths from minimal to maximum and the corresponding
# joint angle list
muscle_para_mf = {}

# extract the property parameters from the OpenSim models
muscle_para_mf['mtu_par_set'] = getMuscleProperties(osim_muscle)

In [10]:
motion_range = []
joints_uniq = []
for joints in wrapping_coordinate[muscle_name]:  # run through all joints           
    for joint in joints:  # extract the motion range
        if joint not in joints_uniq:  # only go through the unique joint coordinates
            joints_uniq.append(joint)
            motion_range.append(getJointRanges_array(osim_model, joint))

In [11]:
mtu_len_set, jit_list_set = getMuscleLengthList(osimModel = osim_model,
                                                osim_muscle = osim_muscle, 
                                                joints = joints_uniq,
                                                ang_ranges = motion_range,
                                                speedy = speedy)

In [12]:
muscle_para_mf['muscle_name'] = muscle_name
muscle_para_mf['mtu_length_osim'] = mtu_len_set
muscle_para_mf['jit_list_set'] = jit_list_set
muscle_para_mf['jit_uniq'] = joints_uniq
muscle_para_mf['jit_uniq_ranges'] = motion_range

In [13]:
# set the muscle activation shift from 0 to 1
# only optimize when activation equal to 1
act_list = [1] # np.linspace(0, 1, 5)
muscle_para_mf['act_list'] = act_list

In [14]:
# get muscle tendon force,  and active/passive forces
mtu_force_length, act_force_length, pas_force_length = getMuscleForceMaps(osim_model, osim_muscle, joints_uniq, jit_list_set, act_list)
    
muscle_para_mf['mtu_force_osim'] = mtu_force_length
muscle_para_mf['act_force_osim'] = act_force_length
muscle_para_mf['pas_force_osim'] = pas_force_length

# Load muscle params

add mujoco unique joint arr information, in case the joint range are different. load saved muscle force data file

In [15]:
# first deep copy the jit list set
muscle_para_mf['mjc_jit_list_set'] = muscle_para_mf['jit_list_set'].copy()

# Test

In [16]:
# then test if the motion range is the same for each unique joint
for i_unique_jit, unique_jit in enumerate(muscle_para_mf['jit_uniq']):
    osim_range = (muscle_para_mf['jit_uniq_ranges'][i_unique_jit][1] - muscle_para_mf['jit_uniq_ranges'][i_unique_jit][0])
    mjc_range = (ang_ranges_mjc[unique_jit][1] - ang_ranges_mjc[unique_jit][0])

    if not (np.abs(osim_range - mjc_range) < 0.005):
        # change all the joint list set of cooresponding joint
        for i_set in range(len(muscle_para_mf['mjc_jit_list_set'])):
            jit_set_list = list(muscle_para_mf['mjc_jit_list_set'][i_set])
    
            jit_set_list[i_unique_jit] = (muscle_para_mf['jit_list_set'][i_set][i_unique_jit] - muscle_para_mf['jit_uniq_ranges'][i_unique_jit][0]) *mjc_range/osim_range + ang_ranges_mjc[unique_jit][0]
            
            muscle_para_mf['mjc_jit_list_set'][i_set] = tuple(jit_set_list)

# Optimize

In [17]:
# Right now only optimize muscle force length relationship when activation equal to 1

# Set all fmax and scale factor as 1, to avoid very large muscle forces
for ig in range(len(mjc_model.actuator_gainprm)):
    mjc_model.actuator_gainprm[ig][2] = 1
    mjc_model.actuator_gainprm[ig][3] = 1

In [18]:
joints_uniq = muscle_para_mf['jit_uniq']     
mjc_jnt_arr = muscle_para_mf['mjc_jit_list_set']
act_arr = muscle_para_mf['act_list']

# Musculotendon length range

In [19]:
# calcualte motion range based on fiber_opt and tendon_slack 
fiber_opt = muscle_para_mf['mtu_par_set']['fiber_opt']
tendon_slack = muscle_para_mf['mtu_par_set']['tendon_sla']

fiber_len_range = np.zeros(2)

# use the osim mtu lengths to calcualte the mujoco muscle parameters
osim_mtu_len_arr = muscle_para_mf['mtu_length_osim']

# if the muscle length is constant, then separate them a bit, since mjc cannot handle equal bounds
if osim_mtu_len_arr[0] == osim_mtu_len_arr[-1]:
    osim_mtu_len_arr[0] = osim_mtu_len_arr[0] - 0.005
    osim_mtu_len_arr[1] = osim_mtu_len_arr[1] + 0.005

# actual muscle length
muscle_inst = mjc_model.actuator(muscle_name)
muscle_inst.lengthrange[0] = osim_mtu_len_arr[0]
muscle_inst.lengthrange[1] = osim_mtu_len_arr[-1]

# Muscle fiber range

In [20]:
# calculate relative fiber lengths
fiber_len_range[0] = (osim_mtu_len_arr[0] - tendon_slack) / fiber_opt
fiber_len_range[1] = (osim_mtu_len_arr[-1] - tendon_slack) / fiber_opt

# make sure that the fiber length range is between [0.01, 1.99], which
# is reasonable
if fiber_len_range[0] < 0.01:
    fiber_len_range[0] = 0.01

if fiber_len_range[1] > 1.99:
    fiber_len_range[1] = 1.99

# operation range (nomalized by L0)
muscle_inst.gainprm[0:2] = fiber_len_range

# set lmin and lmax equal 0.0 and 2.0
muscle_inst.gainprm[4] = 0.0
muscle_inst.gainprm[5] = 2.0

# Force

In [21]:
fmax = muscle_para_mf['mtu_par_set']['fmax']
osimFP = muscle_para_mf['mtu_force_osim'] / fmax

# Error

In [29]:
# function: getMuscleForceDiff
force_mtu, mjc_mtu_length = getMuscleForceLengthCurvesSim(mjc_model = mjc_model,
                                                          muscle = muscle_name,
                                                          joints = joints_uniq,
                                                          jnt_arr = mjc_jnt_arr2,
                                                          act_arr = act_arr)

In [51]:
# biasprm also NEEDS to be updated
muscle_inst.biasprm[:] = muscle_inst.gainprm

# function: getMuscleForceDiff
force_mtu, mjc_mtu_length = getMuscleForceLengthCurvesSim(mjc_model = mjc_model,
                                                          muscle = muscle_name,
                                                          joints = joints_uniq,
                                                          jnt_arr = mjc_jnt_arr,
                                                          act_arr = act_arr)

# change ndarray to array
osim_force = np.asarray(osimFP).ravel()
mjc_force = np.asarray(force_mtu).ravel()

force_error = osim_force + mjc_force
cost_org = np.sqrt(np.mean(force_error ** 2))

# absolute and relative errors
abs_err = abs(force_error)

safe_osim_force = np.maximum(np.abs(osim_force), 1e-3)
rel_err = np.abs(force_error / safe_osim_force)

# check if absolulte error larger than 0.001 and relative error larger than 5%
err_ind = [np.where((abs_err > 0.001) & (rel_err > 0.05))[0].tolist()]

In [52]:
if mjc_mtu_length.min() == mjc_mtu_length.max():
    min_mjc_mtu_length = mjc_mtu_length.min() - 0.005
    max_mjc_mtu_length = mjc_mtu_length.max() + 0.005
else:
    min_mjc_mtu_length = mjc_mtu_length.min()
    max_mjc_mtu_length = mjc_mtu_length.max()
    
muscle_inst.lengthrange[0] = min_mjc_mtu_length
muscle_inst.lengthrange[1] = max_mjc_mtu_length

# Original:
muscle_inst.gainprm[4:6] = [0, 1.8] # original: muscle_inst.gainprm[4:6] = [0, 2], sj: muscle_inst.gainprm[4:6] = [0, 1.8]
muscle_inst.biasprm[4:6] = [0, 1.8] # original: muscle_inst.biasprm[4:6] = [0, 2], sj: muscle_inst.biasprm[4:6] = [0, 1.8]

# save current mujoco model into cvt3...
with open(save_model_path, 'w+') as xml_file:
    mujoco.mj_saveLastXML(save_model_path, mjc_model)

# lower bounds of lmin, lmax, fpmax, fmax
optParam_lb = [0.1, 1, 0.1, 0.5] # original: optParam_lb = [0.01, 1, 0.01, 0.5], sj: optParam_lb = [0.1, 1, 0.1, 0.5]

# upper bounds of lmin, lmax, fpmax, fmax
optParam_ub = [1, 1.6, 1.8, 1.5] # original: optParam_ub = [1, 1.99, 5, 2], sj: optParam_ub = [1, 1.6, 1.8, 1.5]

opt_results, mjc_model = fmOptPSO_cust(mjc_model_path = save_model_path,
                                       muscle = muscle_name,
                                       joints = joints_uniq,
                                       jnt_arr = mjc_jnt_arr,
                                       act_arr = act_arr,
                                       osim_fm = osimFP,
                                       optParam_lb = optParam_lb,
                                       optParam_ub = optParam_ub,
                                       cost_org = cost_org)
muscle_para_mf['opt_results'] = opt_results

# save the data based to the muscle parameter file
with open(muscle_path, 'wb') as muscle_file_saving:
    pickle.dump(muscle_para_mf, muscle_file_saving)

# extract muscle instance again.
muscle_inst = mjc_model.actuator(muscle_name)

# change fmax back to normalized value
if len(muscle_para_mf['opt_results']['res_opt']) > 0:
    muscle_inst.gainprm[2] = muscle_para_mf['opt_results']['res_opt'][3]*fmax
   
else:
    muscle_inst.gainprm[2] = fmax

# change vmax to 10*L0, and fvmax as 1.4 (opensim default)
muscle_inst.gainprm[6] = 10
muscle_inst.gainprm[8] = 1.4

# copy the parameter from gainprm to biasprm
muscle_inst.biasprm = muscle_inst.gainprm

2026-06-11 00:10:14.858 | INFO     | myoconverter.optimization.utils.UtilsForceOpt:fmOptPSO_cust:189 -         PSO iteration: 0 ; 0 percentage similarities; Best obj: 3156.81468
2026-06-11 00:10:14.862 | INFO     | myoconverter.optimization.utils.UtilsForceOpt:fmOptPSO_cust:197 -         Break the optimization, since certain number of similar particles reached


In [53]:
# save the updated model
mujoco.mj_saveLastXML(save_model_path, mjc_model)
print(f"save: {save_model_path}")

save: /mnt/ext1/seojin/temp/myoconverter/cvt3.xml
